# Exploratory Data Analysis — Crowdsourcing Platform

**Thesis:** Quality Evaluation and Worker Segmentation in Crowdsourcing Systems
**Program:** HSE FCS — Master of Data Science (MNAD), 2024–26

---

## 1 · Scope and reading guide

This chapter is a descriptive study of a **one-month snapshot (September 2025)** of a production crowdsourcing platform, covering 4.12 M answers produced by 14 452 workers across 5 projects. Each row corresponds to one worker's answer to one task.

The chapter is **deliberately descriptive**, not predictive. Its purpose is to:

1. document the structural properties of the platform (pools, pages, gold tasks);
2. characterise the workforce (activity, tenure, temporal patterns);
3. characterise the annotation task itself (label balance, error structure);
4. identify **quality signals** that will be formalised as worker-level features in Chapter 4 (Feature Engineering);
5. flag **methodological caveats** that will govern modelling choices in Chapters 5–6.

We deliberately stop short of constructing a worker-level feature table, clustering, or any modelling. Those steps belong to the next chapter. The role of this chapter is to make the data legible.

## 2 · Domain glossary

| Term | Meaning |
|------|---------|
| `pool_type = 0` | **Regular (paid) pool** — production work, workers are paid per task. |
| `pool_type = 1` | **Rehabilitation pool** — control-only; a worker is routed here after making ≥ 2 errors in their last 10 gold tasks. Unpaid. |
| `pool_type = 3` | **Training pool** — onboarding exercises before a worker is allowed into a project. Unpaid. |
| `task_type = 0` | **Regular task** — the actual annotation work. |
| `task_type = 1` | **Gold (control) task** — a task with a known correct answer, injected into regular pools for live quality monitoring. |
| `task_type = 2` | **Training task** — used only inside training pools. |
| Label `1` | Worker answer: *"same / match"*. |
| Label `2` | Worker answer: *"different / mismatch"*. |
| Label `3` | **Service flag** — worker reports that the task itself is malformed. Not a valid class. |
| `page` | A screen shown to a worker; may contain 1–3 tasks. **Timestamps are per page, not per task.** |
| `overlap` | Number of distinct workers who label the same task. |

## 3 · Definitional note: `task_ans` is a *proxy*, not ground truth

Throughout this chapter we repeatedly compare `user_ans` to `task_ans`. It is essential to be clear about what these equalities mean:

- On **gold tasks** (`task_type = 1`), `task_ans` is an **independently verified correct answer** supplied by the platform. Here `user_ans == task_ans` is a genuine accuracy signal.
- On **regular tasks** (`task_type = 0`), `task_ans` is the **platform-aggregated majority answer** produced after overlap. Here `user_ans == task_ans` measures **agreement with the majority**, not with ground truth.

We therefore use two distinct terms throughout the chapter:

- **Gold accuracy** — `user_ans == task_ans` on `task_type = 1`. Independent quality signal.
- **Agreement rate** — `user_ans == task_ans` on `task_type = 0`. Proxy, circular by construction for aggregation evaluation.

Any conclusion that depends on the distinction will be flagged explicitly.

## 4 · Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

# ── palette ───────────────────────────────────────────────────────────
C = dict(
    blue="#4A90D9", teal="#2BA89E", amber="#E5A832", coral="#E06350",
    purple="#7C6BC4", green="#5AAE61", pink="#D96BA0", gray="#8C8C8C",
    red="#D94F4F", slate="#5C6B7A",
)
PROJECT_C = {575:"#7C6BC4", 576:"#4A90D9", 577:"#2BA89E", 578:"#E5A832", 581:"#E06350"}
POOL_C    = {0:C["blue"], 1:C["coral"], 3:C["teal"]}

plt.rcParams.update({
    "figure.facecolor":"white", "axes.facecolor":"white",
    "axes.edgecolor":"#CCCCCC", "axes.linewidth":0.5,
    "axes.grid":True, "grid.color":"#EEEEEE", "grid.linewidth":0.4,
    "font.family":"sans-serif", "font.size":10.5,
    "axes.titlesize":12, "axes.titleweight":600, "axes.titlepad":10,
    "axes.labelsize":10, "axes.labelcolor":"#444444",
    "xtick.color":"#666666", "ytick.color":"#666666",
    "figure.dpi":130, "legend.framealpha":0.85,
    "legend.fontsize":9, "legend.edgecolor":"#DDDDDD",
})

def bar_labels(ax, bars, fmt="{:,.0f}", fs=8.5, pad=3, horizontal=False):
    for b in bars:
        v = b.get_width() if horizontal else b.get_height()
        if horizontal:
            ax.text(v + pad, b.get_y() + b.get_height()/2,
                    fmt.format(v), va="center", fontsize=fs, color="#555")
        else:
            ax.text(b.get_x() + b.get_width()/2, v + pad,
                    fmt.format(v), ha="center", fontsize=fs, color="#555")

def despine(ax, left=False):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if left: ax.spines["left"].set_visible(False)

print("Setup complete")

## 5 · Data loading

In [ ]:
df = pd.read_csv("data.csv", index_col=0)

df["created_at"]  = pd.to_datetime(df["created_at"])
df["finished_at"] = pd.to_datetime(df["finished_at"])

# Page-level duration. Timestamps are per page; per-task time is an approximation.
df["page_duration_sec"] = (df["finished_at"] - df["created_at"]).dt.total_seconds()
df["per_task_sec"]      = df["page_duration_sec"] / df["tasks_per_page"]

POOL_MAP = {0: "Regular", 1: "Rehabilitation", 3: "Training"}
TASK_MAP = {0: "Regular", 1: "Gold (control)", 2: "Training"}
df["pool_label"] = df["pool_type"].map(POOL_MAP)
df["task_label"] = df["task_type"].map(TASK_MAP)

print(f"Loaded {len(df):,} rows  ·  {df['ozon_id'].nunique():,} workers  ·  "
      f"{df['project_id'].nunique()} projects  ·  {df['pool_id'].nunique():,} pools")
print(f"Period: {df['created_at'].dt.date.min()} → {df['created_at'].dt.date.max()}")

In [ ]:
# Missing values
missing = df.isnull().sum()
pct     = (missing / len(df) * 100).round(2)
pd.DataFrame({"missing": missing, "%": pct}).query("missing > 0")

**Missingness is dominated by two mechanisms:**

- `user_ans` (4.14 %) is missing whenever a worker skipped the task or left the page unsubmitted.
- `finished_at` and derived duration columns are missing for the same unsubmitted pages (0.17 %).
- `task_ans` (0.15 %) is missing for a small number of tasks that lack a resolved answer — these are excluded from any correctness analysis.

No imputation is performed; downstream analyses drop rows with the relevant field missing on a case-by-case basis.

## 6 · Preprocessing: removing the `user_ans = 3` service flag

In this platform, `user_ans = 3` is **not a valid class label**. It is a service code the worker can emit to signal *"the task itself is malformed"* (for example, a broken product card, missing image, or ambiguous instructions). Consequently:

- `user_ans = 3` cannot be compared against `task_ans` — the pair is on a different semantic level.
- Leaving these rows in place would inflate both the error rate and the apparent tail of the label distribution, distorting the confusion matrix, per-class accuracy, and the 1 ↔ 2 error analysis.
- The corresponding `task_ans` is never 3 in practice (the platform does not have a "malformed" ground-truth label), which we confirm below.

We first report how frequent the flag is, then drop the affected rows before any correctness computation.

In [ ]:
# ── 6.1 · frequency of the service flag ───────────────────────────────
n_total = len(df)
n_u3    = (df["user_ans"] == 3).sum()
n_u4    = (df["user_ans"] == 4).sum()
n_t3    = (df["task_ans"] == 3).sum()
n_t4    = (df["task_ans"] == 4).sum()

print(f"Total rows:                         {n_total:>12,}")
print(f"Rows with user_ans = 3 (service):   {n_u3:>12,}  ({n_u3/n_total*100:.3f}%)")
print(f"Rows with user_ans = 4 (artefact):  {n_u4:>12,}  ({n_u4/n_total*100:.3f}%)")
print(f"Rows with task_ans = 3:             {n_t3:>12,}  (should be 0 by design)")
print(f"Rows with task_ans = 4:             {n_t4:>12,}")

In [ ]:
# ── 6.2 · drop service-flag rows from the correctness pipeline ───────
# We keep them out of `df_clean`, which is used for all answer-level and
# correctness analyses. The original `df` is preserved for structural
# analyses (pool composition, worker activity, page structure) where the
# service flag is not problematic.

before = len(df)
df_clean = df[~df["user_ans"].isin([3, 4])].copy()
df_clean["correct"] = df_clean["user_ans"] == df_clean["task_ans"]
after = len(df_clean)

print(f"Removed {before - after:,} rows ({(before-after)/before*100:.3f}%)")
print(f"Clean set for correctness analysis: {after:,} rows")

From this point onward:

- **`df`** (4.12 M rows) is used for **structural** analyses where the `user_ans` value is irrelevant: pool composition, page structure, worker activity, lifetime, project breakdown.
- **`df_clean`** (4.12 M rows minus service flags) is used for **all correctness and answer-distribution** analyses: confusion matrices, gold accuracy, agreement rates, per-class balance, temporal accuracy, rehabilitation.

This separation is deliberate and will be referenced wherever the distinction matters.

## 7 · Data structure validation

In [ ]:
ct = pd.crosstab(df["pool_label"], df["task_label"], margins=True)
ct

**The documented pool → task invariants hold exactly:**

- **Regular pools** contain a mixture of regular work (~96 %) and injected gold tasks (~4.2 %). Gold injection is the platform's primary live-quality-monitoring mechanism.
- **Rehabilitation pools** contain *only* gold tasks. A worker routed here must pass enough controls to be reinstated.
- **Training pools** contain *only* training tasks. These never enter the production stream.

These constraints will be used throughout the chapter to decide which pools provide an independent quality signal (gold in regular + rehabilitation) and which pools are purely structural (training).

## 8 · High-level overview

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))

pt = df["pool_label"].value_counts()
bars = axes[0].barh(pt.index, pt.values, height=0.45,
                    color=[POOL_C[k] for k in [0,3,1]])
bar_labels(axes[0], bars, horizontal=True)
axes[0].set_title("Answers by pool type")
axes[0].invert_yaxis(); despine(axes[0])

tt = df["task_label"].value_counts()
bars = axes[1].barh(tt.index, tt.values, height=0.45,
                    color=[C["blue"], C["amber"], C["teal"]])
bar_labels(axes[1], bars, horizontal=True)
axes[1].set_title("Answers by task type")
axes[1].invert_yaxis(); despine(axes[1])

pj = df["project_id"].value_counts().sort_index()
bars = axes[2].bar([str(p) for p in pj.index], pj.values, width=0.55,
                   color=[PROJECT_C[p] for p in pj.index])
bar_labels(axes[2], bars)
axes[2].set_title("Answers by project")
axes[2].set_xlabel("project_id"); despine(axes[2])

plt.tight_layout(); plt.show()

reg = df[df["pool_type"] == 0]
gold_in_reg = reg[reg["task_type"] == 1]
print(f"Gold-task share inside regular pools: {len(gold_in_reg)/len(reg)*100:.1f}%")
print(f"Regular-pool overlap distribution:")
print(reg["overlap"].value_counts().sort_index().to_frame("count"))

## 9 · Page structure and the timing model

A **page** is the unit the worker sees at once. The `page_duration_sec` column we computed at loading time captures **total page time** — if a page contains three tasks, that duration covers all three. For any cross-pool-type comparison we therefore divide by `tasks_per_page`.

In [ ]:
print("Tasks per page by pool type:")
print(pd.crosstab(df["pool_label"], df["tasks_per_page"]))

**Structure:**

- Regular pool: mostly 3 tasks/page (88 %), with a smaller single-task stream (~12 %).
- Rehabilitation: always 2 tasks/page.
- Training: always 1 task/page.

Raw `page_duration_sec` is **not comparable** across pool types without normalisation. We use `per_task_sec = page_duration_sec / tasks_per_page` for the response-time analysis in §13. This is an *average* per task on the page, not a true per-task measurement — the instrumentation does not allow us to separate time spent on task 1 versus task 2 of the same page.

## 10 · Per-project breakdown

In [ ]:
ps = df.groupby("project_id").agg(
    answers   = ("task_id", "size"),
    workers   = ("ozon_id", "nunique"),
    tasks     = ("task_id", "nunique"),
    pools     = ("pool_id", "nunique"),
    med_price = ("price", "median"),
    gold_pct  = ("task_type", lambda x: (x == 1).mean() * 100),
    skip_rate = ("skipped", "mean"),
).round(2)
ps

In [ ]:
pt_proj = pd.crosstab(df["project_id"], df["pool_label"], normalize="index").round(3) * 100
pt_proj.columns.name = None
pt_proj.rename(columns=str).style.format("{:.1f}%")

**Observations:**

- **Project 577** has the widest workforce (11 167 distinct workers) and the highest gold-task share (10.5 %) — onboarding is wide and the quality checks are frequent.
- **Project 578** is the largest by volume (1.04 M answers) but has only 498 workers — a small, specialised team carrying a heavy load.
- **Project 575** is the smallest (190 K answers, 798 workers).
- Cross-project variation in gold-task share (6.7 %–10.5 %) means that the density of the independent quality signal depends on which project the worker is annotating in. This will matter when we build per-worker quality estimates in Chapter 4.

### 10.1 · Worker overlap between projects

In [ ]:
wp = df.groupby("ozon_id")["project_id"].nunique()
ov = wp.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(5, 3))
bars = ax.bar(ov.index.astype(str), ov.values, width=0.5,
              color=[C["gray"], C["blue"], C["purple"], C["amber"], C["coral"]][:len(ov)])
bar_labels(ax, bars)
ax.set_xlabel("# projects"); ax.set_ylabel("Workers"); ax.set_title("Projects per worker")
despine(ax); plt.tight_layout(); plt.show()

multi = wp[wp >= 2].count()
print(f"Workers in 2+ projects: {multi:,} ({multi/len(wp)*100:.1f}%)")

**35 %** of workers participate in two or more projects. This cross-project overlap will be exploited in Chapter 4 when we test whether worker quality generalises across projects, or whether a worker reliable in one project is unreliable in another.

## 11 · Worker activity distribution

In [ ]:
apw = df.groupby("ozon_id").size().rename("n")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

axes[0].hist(apw, bins=np.logspace(0, np.log10(apw.max()), 40),
             color=C["blue"], alpha=0.85, edgecolor="white", linewidth=0.3)
axes[0].set_xscale("log")
axes[0].set_title("Answers per worker (log scale)")
axes[0].set_xlabel("Answers"); axes[0].set_ylabel("Workers"); despine(axes[0])

s = np.sort(apw.values)[::-1]
cum = np.cumsum(s) / s.sum() * 100
x   = np.arange(1, len(s)+1) / len(s) * 100
axes[1].plot(x, cum, color=C["blue"], linewidth=2)
axes[1].axhline(80, color=C["coral"], ls="--", lw=0.8, alpha=0.6)
idx80 = np.searchsorted(cum, 80)
pct80 = x[idx80]
axes[1].axvline(pct80, color=C["coral"], ls="--", lw=0.8, alpha=0.6)
axes[1].annotate(f"{pct80:.0f}% of workers\nproduce 80% of answers",
                 xy=(pct80, 80), fontsize=9, color=C["coral"],
                 xytext=(pct80+12, 62),
                 arrowprops=dict(arrowstyle="->", color=C["coral"], lw=1))
axes[1].set_title("Cumulative answer share (Pareto)")
axes[1].set_xlabel("% of workers (most active first)")
axes[1].set_ylabel("% of answers"); despine(axes[1])

plt.tight_layout(); plt.show()

print(f"Median: {apw.median():.0f}  ·  Mean: {apw.mean():.1f}  ·  Max: {apw.max():,}")
print(f"One-shot workers (1 answer): {(apw==1).sum():,} ({(apw==1).mean()*100:.1f}%)")
print(f"Power workers (>=100): {(apw>=100).sum():,} ({(apw>=100).mean()*100:.1f}%)")

**A pronounced Pareto distribution.** The median worker submits 25 answers; 11.4 % are one-shot. Only 7.5 % of workers (the "power workers" with 100+ answers) account for the bulk of volume.

This has a direct implication for the next chapter: **per-worker quality estimates are statistically meaningful only for a minority of workers.** A feature like "gold accuracy" requires at least several gold tasks to be stable; the long tail of the distribution cannot be scored this way. Any Chapter 4 feature will need a minimum-sample filter.

### 11.1 · Worker lifetime

In [ ]:
wl = df.groupby("ozon_id").agg(t0=("created_at","min"), t1=("created_at","max"), n=("task_id","count"))
wl["hours"] = (wl["t1"] - wl["t0"]).dt.total_seconds() / 3600

active = wl[wl["n"] > 1]

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(active["hours"].clip(upper=500), bins=50, color=C["teal"], alpha=0.85,
        edgecolor="white", linewidth=0.3)
ax.set_title("Worker lifetime (hours, clipped at 500)")
ax.set_xlabel("Hours from first to last answer"); ax.set_ylabel("Workers")
despine(ax); plt.tight_layout(); plt.show()

print(f"One-shot workers: {(wl['n']==1).sum():,} / {len(wl):,}")
print(f"Median lifetime (workers with >=2 answers): {active['hours'].median():.1f} h")
print(f"Max: {wl['hours'].max():.0f} h ({wl['hours'].max()/24:.0f} days)")

Worker lifetimes are short and right-skewed: the median active worker is present for a few hours, while a small tail persists for the full month. This motivates a distinction in Chapter 4 between **transient** workers (one session, little history) and **persistent** workers (multi-day activity, full feature profile).

## 12 · Response time

Because timestamps are recorded at page level, we use `per_task_sec = page_duration_sec / tasks_per_page`. This is an *approximation* and treats time uniformly across the tasks on a page. It is adequate for cross-pool comparisons but should not be read as a precise per-task measurement.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))

for i, (pt, name) in enumerate(POOL_MAP.items()):
    d = df.loc[(df["pool_type"]==pt) & df["per_task_sec"].between(1, 300), "per_task_sec"]
    axes[i].hist(d, bins=60, color=POOL_C[pt], alpha=0.85, edgecolor="white", lw=0.3)
    med = d.median()
    axes[i].axvline(med, color="#333", ls="--", lw=0.8, alpha=0.6)
    axes[i].set_title(f"{name}  (median = {med:.0f} s / task)")
    axes[i].set_xlabel("Seconds per task"); axes[i].set_ylabel("Count")
    despine(axes[i])

plt.suptitle("Per-task response time (1–300 s)", fontsize=12, fontweight=600, y=1.02)
plt.tight_layout(); plt.show()

print("Percentiles - regular pool, per-task seconds:")
d0 = df[(df["pool_type"]==0) & df["per_task_sec"].between(0.5, 1e6)]["per_task_sec"]
for p in [5, 25, 50, 75, 90, 95, 99]:
    print(f"  P{p:>2}: {d0.quantile(p/100):>8.1f}")

- Median per-task time in the regular pool is **11.1 s**, with P5 = 4.6 s and P95 = 231 s.
- The long tail beyond P95 is attributable to idle browser tabs and interrupted sessions rather than genuine task effort, and should be clipped in any downstream feature.
- Rehabilitation and training show similar normalised medians once tasks-per-page is accounted for.

## 13 · Answer distribution and correctness

All correctness computations in this section use `df_clean` (service flag `user_ans = 3` removed in §6). The distinction between **gold accuracy** (independent ground truth) and **agreement rate** (proxy against platform-aggregated answer) is maintained explicitly.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))

# a) answer value distribution
for col, lbl, clr, dx in [("task_ans","Platform answer (proxy/gold)",C["blue"],-0.15),
                            ("user_ans","Worker answer",C["amber"],0.15)]:
    vc = df_clean[col].dropna().value_counts().sort_index()
    axes[0].bar(vc.index + dx, vc.values, width=0.28, color=clr, label=lbl, alpha=0.85)
axes[0].set_xticks([1, 2]); axes[0].legend(fontsize=9)
axes[0].set_title("Answer value distribution (after service-flag removal)")
axes[0].set_xlabel("Answer"); axes[0].set_ylabel("Count")
despine(axes[0])

# b) gold accuracy vs agreement rate
gold_acc_val = (df_clean[(df_clean["task_type"]==1) & df_clean["correct"].notna()]
                ["correct"].mean())
agree_val    = (df_clean[(df_clean["pool_type"]==0) & (df_clean["task_type"]==0)
                         & df_clean["correct"].notna()]["correct"].mean())
rehab_acc    = (df_clean[(df_clean["pool_type"]==1) & df_clean["correct"].notna()]
                ["correct"].mean())

labels_bar = ["Gold acc\n(gold tasks)", "Agreement\n(regular, proxy)", "Rehab gold\n(rehab pool)"]
values_bar = [gold_acc_val, agree_val, rehab_acc]
colors_bar = [C["amber"], C["blue"], C["coral"]]

bars = axes[1].bar(labels_bar, values_bar, width=0.55, color=colors_bar)
for b, v in zip(bars, values_bar):
    axes[1].text(b.get_x()+b.get_width()/2, v+0.005, f"{v:.1%}",
                 ha="center", fontsize=10, fontweight=600)
axes[1].set_title("Gold accuracy vs agreement rate")
axes[1].set_ylabel("Share correct / agreeing")
axes[1].set_ylim(0, 1.05)
axes[1].tick_params(axis="x", labelsize=8)
despine(axes[1])

# c) agreement rate by project (regular pool)
proj_acc = (df_clean[(df_clean["pool_type"]==0) & df_clean["correct"].notna()]
            .groupby("project_id")["correct"].mean().sort_index())
bars = axes[2].bar([str(p) for p in proj_acc.index], proj_acc.values, width=0.5,
                   color=[PROJECT_C[p] for p in proj_acc.index])
for b, v in zip(bars, proj_acc.values):
    axes[2].text(b.get_x()+b.get_width()/2, v+0.003, f"{v:.1%}",
                 ha="center", fontsize=9, fontweight=600)
axes[2].set_title("Agreement rate by project (proxy)")
axes[2].set_ylabel("Agreement rate")
axes[2].set_ylim(0.85, 0.95); despine(axes[2])

plt.tight_layout(); plt.show()

**Observations, with the proxy distinction kept explicit:**

- **Gold accuracy (~83 %)** — computed on `task_type = 1` in both regular and rehabilitation pools. This is the *only* number here that is a true accuracy against an independent ground truth.
- **Agreement rate on regular tasks (~89 %)** — this is **not** accuracy. It measures how often a worker's answer coincides with the platform's aggregated label on the same task. Because the platform label is itself a function of workers' answers (typically via majority vote), this quantity is **circular** when used to evaluate aggregation methods. It remains useful as a descriptive consistency signal.
- **Rehabilitation gold accuracy (~82 %)** is lower than the overall gold accuracy because workers are routed here precisely *after* making errors.
- **Cross-project agreement** (89 %–91 %) shows mild project-level variation — not enough to suggest project-specific models, but enough to justify project-aware normalisation in Chapter 4.

### 13.1 · Class balance per project

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(14, 2.8), sharey=True)

for ax, pid in zip(axes, sorted(df_clean["project_id"].unique())):
    s = df_clean[(df_clean["project_id"]==pid) & df_clean["task_ans"].notna()
                 & (df_clean["pool_type"]==0)]
    vc = s["task_ans"].value_counts(normalize=True).sort_index()
    bars = ax.bar(vc.index.astype(int).astype(str), vc.values,
                  color=PROJECT_C[pid], width=0.5, alpha=0.85)
    for b, v in zip(bars, vc.values):
        if v > 0.01:
            ax.text(b.get_x()+b.get_width()/2, v+0.01, f"{v:.0%}",
                    ha="center", fontsize=8)
    ax.set_title(f"Project {pid}", fontsize=10)
    ax.set_ylim(0, 0.85); despine(ax)

axes[0].set_ylabel("Share")
plt.suptitle("Class balance by project (regular pool, platform labels)",
             fontsize=12, fontweight=600, y=1.03)
plt.tight_layout(); plt.show()

**Note on baseline accuracy.** All projects skew toward label 1 (58 %–73 %). A trivial *"always predict 1"* classifier would achieve that range. Any Chapter 5 aggregation model that does not decisively beat this baseline cannot be claimed as a win. Project 577 has the strongest imbalance (73 % label 1), project 578 the weakest (58 %).

### 13.2 · Error structure — confusion matrix

In [ ]:
main = df_clean[(df_clean["pool_type"]==0) & df_clean["user_ans"].notna()
                & df_clean["task_ans"].notna()]

conf     = pd.crosstab(main["task_ans"], main["user_ans"],
                       rownames=["Platform label"], colnames=["Worker"])
conf_pct = pd.crosstab(main["task_ans"], main["user_ans"], normalize="index").round(3)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

for ax, data, title, fmt in [(axes[0], conf, "Absolute counts", "d"),
                               (axes[1], conf_pct, "Row-normalised", ".1%")]:
    im = ax.imshow(data.values, cmap="Blues", aspect="auto")
    ax.set_xticks(range(len(data.columns))); ax.set_xticklabels(data.columns.astype(int))
    ax.set_yticks(range(len(data.index)));   ax.set_yticklabels(data.index.astype(int))
    ax.set_xlabel("Worker answer"); ax.set_ylabel("Platform label (proxy)"); ax.set_title(title)
    for i in range(len(data)):
        for j in range(len(data.columns)):
            v = data.iloc[i,j]
            txt = f"{v:{fmt}}" if isinstance(v, (int, np.integer)) else f"{v:.1%}"
            ax.text(j, i, txt, ha="center", va="center",
                    color="white" if data.values[i,j] > data.values.max()*0.5 else "#333",
                    fontsize=9)

plt.tight_layout(); plt.show()

**After removing the service flag, the task is strictly binary.** Both axes are now label ∈ {1, 2}, and the residual off-diagonal mass is the genuine 1 ↔ 2 disagreement structure. The confusion matrix supports the modelling choice in Chapter 5: the problem should be formalised as binary classification, and the decision boundary lies between the two dominant classes.

### 13.3 · Gold-task accuracy per worker (independent quality signal)

In [ ]:
# This section uses ONLY gold tasks — the sole independent quality signal
gold = df_clean[(df_clean["task_type"]==1) & df_clean["user_ans"].notna()
                & df_clean["task_ans"].notna()].copy()
gold["hit"] = (gold["user_ans"] == gold["task_ans"]).astype(int)

wg = gold.groupby("ozon_id")["hit"].agg(["mean","count"])
wg.columns = ["gold_acc", "n_gold"]
rel = wg[wg["n_gold"] >= 5]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

axes[0].hist(rel["gold_acc"], bins=20, color=C["amber"], alpha=0.85,
             edgecolor="white", lw=0.3)
axes[0].axvline(rel["gold_acc"].mean(), color="#333", ls="--", lw=0.8,
                label=f"mean = {rel['gold_acc'].mean():.2f}")
axes[0].set_title(f"Gold accuracy distribution (>= 5 gold tasks, n = {len(rel)})")
axes[0].set_xlabel("Gold accuracy"); axes[0].set_ylabel("Workers"); axes[0].legend()
despine(axes[0])

axes[1].scatter(rel["n_gold"], rel["gold_acc"], alpha=0.35, s=15,
                color=C["amber"], edgecolors="none")
axes[1].set_title("Gold accuracy vs sample size")
axes[1].set_xlabel("# gold tasks (log)"); axes[1].set_ylabel("Gold accuracy")
axes[1].set_xscale("log"); despine(axes[1])

plt.tight_layout(); plt.show()

print(f"Gold accuracy — mean: {rel['gold_acc'].mean():.3f}, "
      f"std: {rel['gold_acc'].std():.3f}, "
      f"range: [{rel['gold_acc'].min():.2f}, {rel['gold_acc'].max():.2f}]")

**The first meaningful quality signal.** Among workers with at least five gold tasks, accuracy ranges from ~0.29 to 1.00 with a mean around 0.83. The spread is real, not sampling noise — the right-hand plot shows that the range does not collapse as `n_gold` grows. This heterogeneity is the empirical motivation for worker-level scoring in Chapter 4.

Two caveats for later chapters:
1. The `n_gold >= 5` filter excludes most of the workforce (long-tail activity from §11). Any quality estimate based solely on gold accuracy will have limited coverage.
2. Gold accuracy cannot be computed for transient or low-activity workers. Chapter 4 will need a complementary signal that degrades gracefully when gold data is scarce.

## 14 · Temporal patterns

In [ ]:
df_clean["date"] = df_clean["created_at"].dt.date
df_clean["hour"] = df_clean["created_at"].dt.hour
df_clean["dow"]  = df_clean["created_at"].dt.dayofweek

fig, axes = plt.subplots(2, 2, figsize=(13, 7.5))

# a) daily volume by pool
daily = df_clean.groupby(["date","pool_type"]).size().unstack(fill_value=0)
for pt, nm in POOL_MAP.items():
    if pt in daily.columns:
        axes[0,0].plot(daily.index, daily[pt], label=nm,
                       color=POOL_C[pt], lw=1.5, alpha=0.85)
axes[0,0].set_title("Daily volume by pool type"); axes[0,0].legend()
axes[0,0].set_ylabel("Answers"); axes[0,0].tick_params(axis="x", rotation=45)
despine(axes[0,0])

# b) hourly volume
hr = df_clean.groupby("hour").size()
axes[0,1].bar(hr.index, hr.values, color=C["blue"], alpha=0.85, width=0.7)
axes[0,1].set_title("Hourly distribution")
axes[0,1].set_xlabel("Hour (MSK)"); axes[0,1].set_ylabel("Answers")
despine(axes[0,1])

# c) day of week
dow_names = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
dow_c = df_clean["dow"].value_counts().sort_index()
bars = axes[1,0].bar(dow_names, dow_c.values, width=0.6,
                     color=[C["blue"] if d<5 else C["coral"] for d in range(7)],
                     alpha=0.85)
axes[1,0].set_title("Day of week"); axes[1,0].set_ylabel("Answers")
despine(axes[1,0])

# d) hourly AGREEMENT RATE (not "accuracy" — proxy signal)
ha = (df_clean[(df_clean["pool_type"]==0) & df_clean["correct"].notna()]
      .groupby("hour")["correct"].mean())
axes[1,1].plot(ha.index, ha.values, color=C["coral"], lw=2, marker="o", ms=4)
axes[1,1].fill_between(ha.index, ha.values, ha.min()-0.005, alpha=0.08, color=C["coral"])
axes[1,1].set_title("Hourly agreement rate (regular pool, proxy)")
axes[1,1].set_xlabel("Hour (MSK)"); axes[1,1].set_ylabel("Agreement rate")
axes[1,1].set_ylim(ha.min()-0.01, ha.max()+0.01)
despine(axes[1,1])

plt.tight_layout(); plt.show()

- **Peak hours**: 06:00–14:00 (Moscow time, inferred). The workforce is predominantly morning-oriented.
- **Weekend dip**: Sat–Sun volume drops ~30 % but does not collapse.
- **Agreement rate** shows a mild diurnal cycle (~87–92 %), with a dip around 20:00 consistent with end-of-day fatigue. We deliberately call this "agreement rate", not accuracy, because the evening dip may equally reflect *who works at that hour* as it does *any intrinsic fatigue effect*. The confound cannot be resolved in an EDA.

## 15 · Skip behaviour

In [ ]:
print("Skip rate by pool type:")
for pt, nm in POOL_MAP.items():
    r = df[df["pool_type"]==pt]["skipped"].mean()
    print(f"  {nm:18s}  {r:.2%}")

skip_ids = df[(df["pool_type"]==0) & df["skipped"]]["ozon_id"].unique()
skip_acc = df_clean[(df_clean["ozon_id"].isin(skip_ids)) & (df_clean["pool_type"]==0)
                    & df_clean["correct"].notna()]["correct"].mean()
no_skip_acc = df_clean[(~df_clean["ozon_id"].isin(skip_ids)) & (df_clean["pool_type"]==0)
                       & df_clean["correct"].notna()]["correct"].mean()

print(f"\nWorkers who ever skipped: {len(skip_ids):,}")
print(f"Agreement rate — skippers:     {skip_acc:.4f}")
print(f"Agreement rate — non-skippers: {no_skip_acc:.4f}")
print(f"Difference:                    {skip_acc - no_skip_acc:+.4f}")

- Skip rate varies structurally by pool: 0 % in training (mandatory), ~1 % in rehabilitation (workers are motivated to be reinstated), ~4.5 % in regular pools.
- **Workers who ever skip** show an agreement rate ~3.5 pp lower than workers who never skip. This is a descriptive association, not a causal claim — skipping and agreement rate may both reflect a latent "engagement" factor.
- Skip rate is retained as a candidate feature for Chapter 4.

## 16 · Quality heterogeneity among rated workers

For workers with a sufficient gold-task sample (>= 5 gold, and at least a minimal regular-task footprint for comparability), how does gold accuracy relate to other observable quantities? This is the final descriptive step before Chapter 4 begins constructing features.

In [ ]:
# Compact per-worker summary — NOT a feature table.
# This is a descriptive slice of the data, not the Chapter 4 feature construction.

gold_per_w = (df_clean[(df_clean["task_type"]==1) & df_clean["correct"].notna()]
              .groupby("ozon_id")["correct"]
              .agg(gold_acc="mean", n_gold="count"))

dur_per_w = (df_clean[df_clean["per_task_sec"].between(1, 600)]
             .groupby("ozon_id")["per_task_sec"].median()
             .rename("med_duration"))

ent_per_w = (df_clean[df_clean["user_ans"].notna()]
             .groupby("ozon_id")["user_ans"]
             .apply(lambda s: -(s.value_counts(normalize=True)
                                  * np.log2(s.value_counts(normalize=True).clip(lower=1e-10))).sum())
             .rename("answer_entropy"))

n_per_w = df_clean.groupby("ozon_id").size().rename("n_answers")

slice_df = (gold_per_w.join(dur_per_w, how="left")
                      .join(ent_per_w, how="left")
                      .join(n_per_w, how="left"))

rw = slice_df[(slice_df["n_gold"] >= 5) & (slice_df["n_answers"] >= 10)].copy()
print(f"Rated workers (>= 5 gold, >= 10 answers): {len(rw):,}")

In [ ]:
if len(rw) < 20:
    print(f"Only {len(rw)} rated workers in this sample — "
          "heterogeneity plots are meaningful only on the full dataset.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    # a) gold accuracy distribution (the headline heterogeneity plot)
    axes[0].hist(rw["gold_acc"], bins=25, color=C["purple"], alpha=0.85,
                 edgecolor="white", lw=0.3)
    axes[0].axvline(rw["gold_acc"].mean(), color="#333", ls="--", lw=0.8,
                    label=f"mean = {rw['gold_acc'].mean():.2f}")
    axes[0].set_xlabel("Gold accuracy"); axes[0].set_ylabel("Workers")
    axes[0].set_title("Gold accuracy spread (rated workers)"); axes[0].legend()
    despine(axes[0])

    # b) speed vs quality
    sc = axes[1].scatter(rw["med_duration"].clip(upper=150), rw["gold_acc"],
                         alpha=0.35, s=18, c=np.log1p(rw["n_answers"]),
                         cmap="viridis", edgecolors="none")
    axes[1].set_xlabel("Median per-task time (s)"); axes[1].set_ylabel("Gold accuracy")
    axes[1].set_title("Speed vs gold accuracy (colour = log activity)")
    plt.colorbar(sc, ax=axes[1], shrink=0.8)
    despine(axes[1])

    # c) entropy vs quality
    axes[2].scatter(rw["answer_entropy"], rw["gold_acc"], alpha=0.35, s=18,
                    color=C["amber"], edgecolors="none")
    axes[2].set_xlabel("Answer entropy"); axes[2].set_ylabel("Gold accuracy")
    axes[2].set_title("Answer entropy vs gold accuracy")
    despine(axes[2])

    plt.tight_layout(); plt.show()

**Workers are not homogeneous.** Gold accuracy among rated workers spans roughly 0.3 to 1.0 with a mean near 0.83. Three qualitative regions are visible in the scatter plots:

1. **High gold accuracy, moderate entropy** — reliable annotators who adapt their answers to the task.
2. **Low gold accuracy, low entropy** — candidate spammers who always pick the same label; identifiable from the combination of low accuracy and answer distribution concentrated on one value.
3. **Low gold accuracy, high entropy** — workers whose answers are varied but frequently wrong; potentially struggling or random.

These are **qualitative groupings suggested by the scatter plots**, not clusters. Formal clustering is performed in Chapter 4. The purpose here is only to motivate it: the heterogeneity is real and large enough that a single global model of worker behaviour would be inappropriate.

## 17 · Rehabilitation effectiveness

Workers are routed to rehabilitation after making ≥ 2 errors in the last 10 gold tasks. The platform intends this as a corrective mechanism. Does it work?

This question is intrinsically *causal* and cannot be fully answered with observational data. We report descriptive before/after statistics, a non-parametric test, and then explicitly list the confounders that prevent a causal interpretation.

In [ ]:
rehab_ids = df_clean[df_clean["pool_type"]==1]["ozon_id"].unique()

results = []
for wid in rehab_ids:
    ws = df_clean[df_clean["ozon_id"]==wid].sort_values("created_at")
    t0 = ws[ws["pool_type"]==1]["created_at"].min()

    before = ws[(ws["pool_type"]==0) & (ws["created_at"] < t0) & ws["correct"].notna()]
    after  = ws[(ws["pool_type"]==0) & (ws["created_at"] > t0) & ws["correct"].notna()]

    if len(before) >= 5 and len(after) >= 5:
        results.append({
            "acc_before": before["correct"].mean(),
            "acc_after":  after["correct"].mean(),
            "n_before":   len(before),
            "n_after":    len(after),
            "delta":      after["correct"].mean() - before["correct"].mean(),
        })

rdf = pd.DataFrame(results)

if len(rdf) < 10:
    print(f"Only {len(rdf)} workers with sufficient before/after data — "
          "not enough for a rehabilitation analysis on this sample.")
    print("This section produces meaningful results only on the full dataset.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    bins = np.linspace(0.4, 1.0, 25)
    axes[0].hist(rdf["acc_before"], bins=bins, alpha=0.55, color=C["coral"],
                 edgecolor="white", lw=0.3, label="Before rehab")
    axes[0].hist(rdf["acc_after"], bins=bins, alpha=0.55, color=C["teal"],
                 edgecolor="white", lw=0.3, label="After rehab")
    axes[0].axvline(rdf["acc_before"].mean(), color=C["coral"], ls="--", lw=1.2)
    axes[0].axvline(rdf["acc_after"].mean(),  color=C["teal"],  ls="--", lw=1.2)
    axes[0].set_xlabel("Agreement rate (regular pool, proxy)")
    axes[0].set_ylabel("Workers")
    axes[0].set_title("Before vs after rehabilitation")
    axes[0].legend(); despine(axes[0])

    rdf["before_bin"] = pd.cut(
        rdf["acc_before"],
        bins=[0, 0.6, 0.7, 0.8, 0.9, 1.01],
        labels=["<60%", "60-70%", "70-80%", "80-90%", "90-100%"])
    bucket = rdf.groupby("before_bin", observed=True)["delta"].agg(["mean","count"])

    bars = axes[1].bar(
        bucket.index.astype(str), bucket["mean"],
        color=[C["coral"] if v < 0 else C["teal"] for v in bucket["mean"]],
        alpha=0.85, width=0.6)
    for b, (_, r) in zip(bars, bucket.iterrows()):
        y = b.get_height()
        axes[1].text(b.get_x()+b.get_width()/2,
                     y + (0.004 if y >= 0 else -0.009),
                     f"n={r['count']:.0f}", ha="center", fontsize=8)
    axes[1].axhline(0, color="#333", ls="-", lw=0.5, alpha=0.3)
    axes[1].set_xlabel("Agreement rate before rehab")
    axes[1].set_ylabel("Mean delta")
    axes[1].set_title("Who benefits from rehabilitation?")
    despine(axes[1])

    plt.tight_layout(); plt.show()

    print(f"Workers with sufficient before/after data: {len(rdf)}")
    print(f"Mean before: {rdf['acc_before'].mean():.4f}")
    print(f"Mean after:  {rdf['acc_after'].mean():.4f}")
    print(f"Mean delta:  {rdf['delta'].mean():+.4f}")
    print(f"Improved: {(rdf['delta']>0).sum()} ({(rdf['delta']>0).mean()*100:.0f}%)")
    stat, pval = stats.wilcoxon(rdf["acc_before"], rdf["acc_after"])
    print(f"Wilcoxon signed-rank test: W = {stat:.0f}, p = {pval:.4f}")

**Descriptive finding.** Among 765 workers with ≥ 5 answers on either side of their first rehabilitation visit, the mean agreement rate rises from **82.7 %** to **83.9 %** (delta = **+1.24 pp**, Wilcoxon p = 0.002). The effect is strongest for workers starting below 60 % and near zero for workers above 90 %.

**This is a descriptive association, not a causal claim.** Three confounders prevent causal interpretation:

1. **Regression to the mean.** Workers enter rehabilitation precisely after a streak of errors. Even without any intervention, their subsequent performance would partially revert to their individual baseline — and the bucketed chart shows exactly the signature of this effect.
2. **Survivorship bias.** Workers who quit after rehabilitation are excluded (no "after" data). Those who remain in the dataset are self-selected for resilience or engagement.
3. **Proxy quality signal.** We compare agreement rate on regular tasks, not gold accuracy — the proxy may itself drift with a worker's experience.

A proper causal evaluation would require either a randomised intervention or a natural experiment (e.g. a threshold-based regression discontinuity). We surface the descriptive result here because it is a structural property of the platform worth knowing, but we do **not** use it to justify any modelling choice in later chapters.

## 18 · Summary and transition to Chapter 4

### Structural facts about the data

| # | Fact | Source §  |
|---|------|-----------|
| 1 | 4.12 M answers · 14 452 workers · 5 projects · one-month window | §5, §8 |
| 2 | Three pool types with strict task-type constraints: regular mixes regular + gold, rehabilitation is gold-only, training is training-only | §7 |
| 3 | Timestamps are per-page, not per-task. Per-task time is an approximation | §9 |
| 4 | `user_ans = 3` is a service flag ("malformed task"), not a class; removed from all correctness analysis | §6 |
| 5 | The annotation task is effectively binary (labels 1 and 2) after service-flag removal | §13.2 |
| 6 | Class balance is skewed: 58 %–73 % label 1, so trivial baselines score in that range | §13.1 |

### Quality signals the chapter exposes

| Signal | Type | Coverage | Where it becomes a feature |
|--------|------|----------|---------------------------|
| Gold accuracy | Independent (ground truth) | Limited: needs `n_gold >= 5`, covers ~1k workers | Primary Chapter 4 feature |
| Agreement rate on regular tasks | Proxy against platform majority | Broad: nearly all regular-pool workers | Secondary feature, with circularity warning for Chapter 5 |
| Per-task response time | Behavioural | Broad | Candidate feature, weak correlation with quality |
| Answer entropy / label concentration | Behavioural | Broad | Candidate feature, distinguishes spammers from struggling workers |
| Skip rate | Engagement | Broad | Candidate feature, descriptively associated with lower agreement |

### What this chapter deliberately did *not* do

- It did not build a worker-level feature table.
- It did not cluster workers.
- It did not compare aggregation models.
- It did not run correlation matrices over engineered features.
- It did not report speed/overlap/price "effects on quality" without confounders.

All of the above belong to Chapter 4 (Feature Engineering and Baselines) and Chapter 5 (Advanced Models and Segmentation). Separating the descriptive chapter from the modelling chapters keeps the methodological commitments clean: every claim in this chapter is an empirical description of the data; every claim in the next chapter will rest on an explicit modelling choice.

### What Chapter 4 will do with this

1. **Construct a worker-level feature table** with activity, timing, behavioural, and contextual features. Gold accuracy and agreement rate will both be included, but they will be treated as different kinds of signal.
2. **Define a minimum-data threshold** for per-worker quality estimates, based on the sample-size analysis of §13.3.
3. **Build baseline aggregation methods** (majority vote, weighted MV, Dawid–Skene) and evaluate them *against gold accuracy*, not against platform agreement — the circularity identified in §13 forces this choice.
4. **Motivate worker segmentation** using the qualitative heterogeneity regions visible in §16.
5. **Acknowledge the rehabilitation finding as descriptive only** and not depend on it for any modelling claim.